# Benchmark Plotting — Distribution Statistics

This notebook consolidates descriptive statistics for every available cleaned economic series. It reads the canonical `data/cleaned/*.csv` files generated by `dataCleansing.py` and summarizes the cleaned `value` levels (not delta-transformed values).

Statistics: mean, median, sample standard deviation, and sample variance (`ddof=1`). Each series stays at its native frequency and native unit. Because units and scales differ, cross-series comparisons of raw-level dispersion are not directly meaningful. Missing/non-numeric values are excluded by pandas; no resampling, imputation, transformations, or outlier filtering are performed here.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Resolve paths from the repository root or from the notebook/ directory.
candidates = [Path('data/cleaned'), Path('../data/cleaned')]
cleaned_dir = next((p for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Could not find data/cleaned. Run dataCleansing.py first or open this notebook from the repository root/notebook directory.')

csv_paths = sorted(cleaned_dir.glob('*.csv'))
csv_paths = [p for p in csv_paths if p.name != 'series_metadata.csv']
if not csv_paths:
    raise FileNotFoundError(f'No cleaned series CSV files found in {cleaned_dir.resolve()}')

rows = []
for path in csv_paths:
    series_id = path.stem
    df = pd.read_csv(path)
    if 'value' not in df.columns:
        continue
    values = pd.to_numeric(df['value'], errors='coerce').dropna()
    rows.append({
        'series_id': series_id,
        'n': int(values.count()),
        'mean': values.mean() if len(values) else float('nan'),
        'median': values.median() if len(values) else float('nan'),
        'std_sample': values.std(ddof=1) if len(values) >= 2 else float('nan'),
        'variance_sample': values.var(ddof=1) if len(values) >= 2 else float('nan'),
        'data_file': path.as_posix(),
    })

stats_table = pd.DataFrame(rows).sort_values('series_id').reset_index(drop=True)
stats_table[['mean', 'median', 'std_sample', 'variance_sample']] = stats_table[['mean', 'median', 'std_sample', 'variance_sample']].round(6)
display(stats_table)
print(f'Series summarized: {len(stats_table)}')

## Export

The CSV export is a review artifact of the current cleaned snapshot. It does not alter cleaned inputs or create a benchmark model.

In [ ]:
output_dir = Path('outputs/benchmark') if Path('outputs').exists() else Path('../outputs/benchmark')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'cleaned_level_distribution_statistics.csv'
stats_table.to_csv(output_path, index=False)
print(f'Saved: {output_path.resolve()}')